# Advanced 01 — Corrective RAG: Bounded Recovery After Retrieval Failure

**Scenario:** an Acme support-policy assistant must recover from weak evidence without widening authorization or searching indefinitely.  
**Runtime:** deterministic, local, and credential-free by default; optional live grading and generation use a configured model.  
**Primary rule:**

> Corrective RAG is a bounded evidence-recovery controller. When the available authorized evidence is inadequate, choose only from policy-approved recovery routes; if those routes cannot establish sufficient evidence within budget, abstain.

We keep one scenario and progressively build: real retrieval → typed grading → failure-specific recovery → re-grading → terminal decisions → evaluation → the same controller in LangGraph.


## Learning objectives, success criteria, and boundaries

By the end you can:

- inspect a real dense first-stage retrieval and BM25 fallback;
- combine deterministic eligibility checks with semantic evidence grading;
- distinguish document-level relevance from set-level sufficiency;
- route lexical, semantic, partial-coverage, stale, corpus-gap, conflict, authorization, and clarification outcomes;
- enforce attempts, rewrite, source, and route budgets;
- preserve request-local evidence IDs and source provenance;
- compare fixed RAG and CRAG on the same labelled cases; and
- express the same finite controller with low-level LangGraph primitives.

**Success criteria:** strong evidence bypasses recovery; every recovery returns to grading; unauthorized scope is never widened; unresolved conflict never generates; a failed route terminates within budget; citations resolve to eligible evidence.

**Non-goals:** this is not an open-ended agent, an internet-search tutorial, a production authorization service, or an exact reproduction of the CRAG paper. The external route uses an approved synthetic corpus to make the trust-boundary change inspectable.

**Prerequisites:** [Retrieval Strategies](../../intermediate/01-retrieval-strategies/README.md), [Metadata and Permissions](../../intermediate/02-metadata-permissions/README.md), and [Evaluation](../../intermediate/04-evaluation/README.md).


## Controller architecture

![Authorized retrieval flows to an evidence evaluator and policy controller, which accepts, abstains, or uses bounded recovery routes that loop back to evaluation.](assets/corrective-control-loop.svg)

The evaluator describes evidence failure. Deterministic policy decides which route is permitted. A route is only a hypothesis: recovered evidence must pass the same grade before generation.


## 1. Environment and reproducibility

Install the learner dependencies with `pip install -e '.[learner]'`. The first-stage retriever below is real local retrieval: transparent normalized unigram/bigram vectors for cosine search and `rank-bm25` for lexical search. The teaching vectorizer deliberately omits identifier-shaped tokens, creating a realistic reason to add lexical recovery without turning this lesson into another vector-database tutorial.

The notebook uses frozen, inspectable grading and generation fixtures by default. Set `CRAG_USE_LIVE_GRADER=1` or `CRAG_USE_LIVE_GENERATOR=1` with `OPENAI_API_KEY` and `CRAG_MODEL` to opt into structured model calls. All core experiments and assertions run offline without explicit opt-in.


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
import math
import os
import re
from statistics import median
from time import perf_counter
from typing import Literal, TypedDict

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pydantic import BaseModel
from rank_bm25 import BM25Okapi
from langgraph.graph import END, START, StateGraph

print({
    "live_grader_requested": os.getenv("CRAG_USE_LIVE_GRADER") == "1",
    "live_generator_requested": os.getenv("CRAG_USE_LIVE_GENERATOR") == "1",
})


## 2. Typed contracts: identity, evidence, grade, policy, and trace

The routing contract is categorical rather than a vague confidence score. `EvidenceGrade` records what is covered, what is missing, and why recovery is needed. Authorization, lifecycle rules, route allowlists, and budgets remain application-owned invariants.


In [ ]:
GradeState = Literal["strong", "weak", "insufficient"]
FailureType = Literal[
    "lexical_gap", "semantic_gap", "partial_coverage", "stale", "conflict",
    "corpus_gap", "authorization_limited", "underspecified",
]
TerminalState = Literal[
    "answered", "insufficient_evidence", "insufficient_authorized_evidence",
    "conflicting_evidence", "clarification_required", "budget_exhausted",
]


class EvidenceGrade(BaseModel):
    state: GradeState
    failure_types: list[FailureType]
    rationale: str
    covered_requirements: list[str]
    missing_requirements: list[str]


class GroundedAnswer(BaseModel):
    text: str
    citation_ids: list[str]
    unresolved_conflicts: list[str] = []


@dataclass(frozen=True)
class Principal:
    user_id: str
    tenant_id: str
    clearance: str = "internal"


@dataclass(frozen=True)
class EvidenceChunk:
    text: str
    metadata: dict


@dataclass(frozen=True)
class EvidenceItem:
    evidence_id: str
    chunk: EvidenceChunk
    route: str
    retrieval_query: str
    parent_evidence_id: str | None = None
    selected_span: str | None = None


@dataclass(frozen=True)
class RecoveryPolicy:
    max_attempts: int = 2
    max_rewrites: int = 1
    external_allowed: bool = True
    allowed_routes: frozenset[str] = frozenset({
        "lexical_fallback", "query_rewrite", "targeted_retrieval",
        "fresh_source", "external_source", "clarification",
    })


@dataclass
class RuntimeCounters:
    retrieval_calls: int = 0
    grader_calls: int = 0
    rewrite_calls: int = 0
    external_source_calls: int = 0
    generation_calls: int = 0
    unauthorized_route_attempts: int = 0
    elapsed_ms: float = 0.0


@dataclass(frozen=True)
class EvalCase:
    case_id: str
    slice: str
    query: str
    requirements: tuple[str, ...]
    expected_initial_grade: GradeState
    expected_route: str
    expected_terminal_state: TerminalState
    required_evidence_ids: tuple[str, ...]
    principal_tenant: str = "acme"
    external_allowed: bool = True
    initial_k: int = 2


@dataclass
class ControllerResult:
    case_id: str
    initial_grade: EvidenceGrade
    final_grade: EvidenceGrade
    terminal_state: TerminalState
    evidence: list[EvidenceItem]
    attempts: int
    route_history: list[str]
    ledger: dict
    counters: RuntimeCounters
    answer: dict | None


## 3. Build a controlled enterprise evidence corpus

The 38 synthetic chunks span Acme, Globex, NovaTech, and an approved public source. They include same-domain distractors, exact identifiers, semantic paraphrases, compound evidence, historical/current versions, unresolved conflict, long mixed-content passages, external-only facts, and another tenant's restricted project. No credentials or secrets are indexed.

Every chunk has stable `document_id`, `chunk_id`, `source`, `tenant_id`, `status`, `effective_date`, and `authority` metadata. `covers` and optional fact metadata are teaching labels used by the frozen evaluator and evaluation harness—not fields a production model magically knows.


In [ ]:
def chunk(
    document_id: str,
    part: str,
    text: str,
    *,
    tenant_id: str = "acme",
    status: str = "current",
    effective_date: str = "2026-01-01",
    authority: str = "service_owner",
    classification: str = "internal",
    covers: tuple[str, ...] = (),
    fact_key: str | None = None,
    fact_value: str | None = None,
    source_type: str = "internal",
    selected_spans: dict[str, str] | None = None,
) -> EvidenceChunk:
    return EvidenceChunk(
        text=text,
        metadata={
            "document_id": document_id,
            "chunk_id": f"{document_id}#{part}",
            "source": f"{document_id}.md",
            "tenant_id": tenant_id,
            "status": status,
            "effective_date": effective_date,
            "authority": authority,
            "classification": classification,
            "covers": list(covers),
            "fact_key": fact_key,
            "fact_value": fact_value,
            "source_type": source_type,
            "selected_spans": selected_spans or {},
        },
    )


internal_corpus = [
    chunk("acme-checkout-runbook", "gateway", "Acme checkout uses the Adyen gateway for card authorization.", covers=("checkout_gateway",)),
    chunk("acme-checkout-runbook", "failover", "To fail over checkout, freeze writes, verify replica lag, promote the warm replica, then reopen traffic in ten-percent steps.", covers=("failover_steps",)),
    chunk("acme-checkout-runbook", "approval", "Checkout failover requires approval from the Database Reliability lead and Incident Commander.", covers=("failover_approver",)),
    chunk("acme-parts-catalog", "ax774", "Replacement assembly AX-774-B is the approved actuator for kiosk model K-9.", covers=("ax774_replacement",)),
    chunk("acme-parts-catalog", "rt991", "Repair token RT-991-C applies to the west-region barcode controller.", covers=("rt991_controller",)),
    chunk("acme-parts-catalog", "zx204", "Module ZX-204 requires calibration profile Delta-7 after installation.", covers=("zx204_calibration",)),
    chunk("acme-incident-policy-v1", "severity", "The retired 2024 policy declared checkout Severity 1 after thirty minutes of total outage.", status="historical", effective_date="2024-01-01", authority="policy_board", covers=("severity_threshold",), fact_key="severity_threshold", fact_value="30 minutes"),
    chunk("acme-incident-policy-v2", "severity", "The current policy declares checkout Severity 1 after fifteen minutes of total outage.", authority="policy_board", covers=("severity_threshold",), fact_key="severity_threshold", fact_value="15 minutes"),
    chunk("acme-change-calendar", "window-a", "The current operations calendar assigns database maintenance to Sunday 02:00 UTC.", authority="operations_board", covers=("maintenance_window",), fact_key="maintenance_window", fact_value="Sunday 02:00 UTC"),
    chunk("acme-database-standard", "window-b", "The current database standard assigns database maintenance to Saturday 23:00 UTC.", authority="database_board", covers=("maintenance_window",), fact_key="maintenance_window", fact_value="Saturday 23:00 UTC"),
    chunk("acme-dr-guide", "replica", "When the primary data plane becomes unavailable, elevate the warm replica, redirect the writer endpoint, and validate reconciliation checkpoints.", covers=("dr_switch",)),
    chunk("acme-support-guide", "priority", "A customer-impacting checkout incident is escalated to Priority A after impact confirmation.", covers=("priority_rule",)),
    chunk("acme-refund-policy", "approval", "Large refunds over 10,000 dollars require Finance Director approval.", covers=("refund_approver",)),
    chunk("acme-refund-policy", "long", "Background: refund requests arrive through several channels. Training notes describe dashboard colors. Policy: refunds above 10,000 dollars require Finance Director approval. Historical anecdotes are not decision criteria.", covers=("refund_approver",), selected_spans={"refund_approver": "Refunds above 10,000 dollars require Finance Director approval."}),
    chunk("acme-sla", "enterprise", "Enterprise customers receive a 30-minute Priority A response target.", covers=("enterprise_sla",)),
    chunk("acme-sla", "standard", "Standard customers receive a four-hour Priority A response target.", covers=("standard_sla",)),
    chunk("acme-audit-policy", "retention", "Checkout change audit records are retained for seven years.", covers=("audit_retention",)),
    chunk("acme-residency-policy", "canada", "Canadian customer support exports must remain in the Canada region.", covers=("canada_residency",)),
    chunk("acme-oncall", "escalation", "Page the payments on-call after two failed checkout probes.", covers=("oncall_escalation",)),
    chunk("acme-release-policy", "rollback", "Rollback begins when checkout error rate exceeds five percent for ten minutes.", covers=("rollback_trigger",)),
    chunk("acme-capacity-guide", "queue", "Scale checkout workers when queue depth exceeds 5,000 for five minutes.", covers=("scale_trigger",)),
    chunk("acme-mobile-guide", "sync", "Mobile sync recovery clears only acknowledged local queue entries.", covers=("mobile_sync",)),
    chunk("globex-checkout-runbook", "failover", "Globex checkout failover promotes its Frankfurt replica after the regional commander approves.", tenant_id="globex", covers=("globex_failover",)),
    chunk("globex-project-orion", "launch", "Project Orion launches during the third weekend of November.", tenant_id="globex", classification="restricted", covers=("orion_launch",)),
    chunk("globex-parts", "ax774", "Globex uses assembly AX-774-B only for warehouse gate model G-4.", tenant_id="globex", covers=("globex_ax774",)),
    chunk("globex-sla", "enterprise", "Globex enterprise response target is forty-five minutes.", tenant_id="globex", covers=("globex_sla",)),
    chunk("novatech-mobile", "recovery", "NovaTech recovers offline orders by replaying signed sync envelopes.", tenant_id="novatech", covers=("novatech_recovery",)),
    chunk("novatech-devices", "approval", "NovaTech device policy changes require Security Operations approval.", tenant_id="novatech", covers=("novatech_device_approval",)),
    chunk("acme-observability", "traces", "Checkout traces retain route, evidence identifiers, and terminal reason codes.", covers=("trace_fields",)),
    chunk("acme-cache-policy", "scope", "Retrieval cache keys include tenant, policy version, and collection version.", covers=("cache_scope",)),
    chunk("acme-vendor-policy", "authority", "Vendor advisories are supporting evidence and do not override Acme policy without review.", covers=("vendor_authority",)),
    chunk("acme-rehearsal", "owner", "The failover rehearsal owner is the Resilience Engineering lead.", covers=("rehearsal_owner",)),
    chunk("acme-vendor-waiver", "approver", "A vendor waiver requires approval from the Procurement Risk lead.", covers=("waiver_approver",)),
]

external_corpus = [
    chunk("approved-nimbus-advisory", "eos", "Nimbus-Edge version 3 reaches end of support on 31 December 2026.", tenant_id="public", authority="approved_vendor", classification="public", covers=("nimbus_eos",), source_type="external"),
    chunk("approved-regulator-bulletin", "r17", "Regulation R-17 revision 4 takes effect on 1 October 2026.", tenant_id="public", authority="approved_regulator", classification="public", covers=("r17_effective_date",), source_type="external"),
    chunk("approved-status-archive", "region", "The public status archive reports a west-region networking incident on 18 August 2026.", tenant_id="public", authority="approved_status", classification="public", covers=("west_incident_date",), source_type="external"),
    chunk("approved-vendor-guide", "profile", "Vendor guidance confirms calibration profile Delta-7 for module ZX-204.", tenant_id="public", authority="approved_vendor", classification="public", covers=("zx204_calibration",), source_type="external"),
    chunk("unrelated-public-note", "conference", "The annual support engineering conference opens in Vancouver in June.", tenant_id="public", authority="public_notice", classification="public", covers=("conference_date",), source_type="external"),
]

corpus_df = pd.DataFrame([{**c.metadata, "text": c.text} for c in internal_corpus + external_corpus])
display(corpus_df[["tenant_id", "chunk_id", "status", "authority", "source_type"]].head(12))
print("internal chunks:", len(internal_corpus), "external chunks:", len(external_corpus))
assert 25 <= len(internal_corpus + external_corpus) <= 40
assert corpus_df.chunk_id.is_unique


## 4. Real first-stage retrieval and trusted eligibility

The local dense baseline maps tokens and bigrams into a normalized sparse vector and performs cosine ranking. It intentionally omits identifier-shaped tokens from that representation; BM25 preserves them. This makes the lexical recovery route a measured architectural complement rather than a hard-coded query branch.

`eligible_internal` derives tenant/classification scope from `Principal`, never from query text. The authorization probe returns only a Boolean diagnostic that matching evidence was excluded; it never returns forbidden content to controller state.


In [ ]:
def lexical_tokens(text: str) -> list[str]:
    return re.findall(r"[a-z]+-?\d+(?:-[a-z0-9]+)*|[a-z]+", text.lower())


DENSE_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "does", "do", "for",
    "from", "how", "in", "is", "it", "its", "of", "on", "or", "the",
    "to", "what", "when", "which", "who", "with",
}


def dense_tokens(text: str) -> list[str]:
    tokens = [
        token for token in lexical_tokens(text)
        if token not in DENSE_STOPWORDS and not any(ch.isdigit() for ch in token)
    ]
    return tokens + [f"{a}_{b}" for a, b in zip(tokens, tokens[1:])]


def dense_vector(text: str) -> dict[str, float]:
    # A transparent local semantic baseline: normalized unigram/bigram features.
    # It intentionally cannot resolve every vocabulary mismatch; recovery must do that.
    counts = Counter(dense_tokens(text))
    norm = math.sqrt(sum(value * value for value in counts.values())) or 1.0
    return {token: value / norm for token, value in counts.items()}


def cosine(left: dict[str, float], right: dict[str, float]) -> float:
    return sum(value * right.get(token, 0.0) for token, value in left.items())


class LocalRetrievers:
    def __init__(self, internal: list[EvidenceChunk], external: list[EvidenceChunk]):
        self.internal = internal
        self.external = external
        self._internal_vectors = {c.metadata["chunk_id"]: dense_vector(c.text) for c in internal}
        self._external_bm25 = BM25Okapi([lexical_tokens(c.text) for c in external])

    @staticmethod
    def eligible_internal(principal: Principal, chunk_: EvidenceChunk, *, status: str | None = None) -> bool:
        allowed = chunk_.metadata["tenant_id"] == principal.tenant_id
        allowed = allowed and chunk_.metadata["classification"] in {"public", principal.clearance}
        return allowed and (status is None or chunk_.metadata["status"] == status)

    def dense(self, query: str, principal: Principal, *, k: int = 2, status: str | None = None) -> list[EvidenceChunk]:
        query_vector = dense_vector(query)
        candidates = [c for c in self.internal if self.eligible_internal(principal, c, status=status)]
        return sorted(candidates, key=lambda c: cosine(query_vector, self._internal_vectors[c.metadata["chunk_id"]]), reverse=True)[:k]

    def lexical(self, query: str, principal: Principal, *, k: int = 3, status: str | None = None) -> list[EvidenceChunk]:
        candidates = [c for c in self.internal if self.eligible_internal(principal, c, status=status)]
        index = BM25Okapi([lexical_tokens(c.text) for c in candidates])
        scores = index.get_scores(lexical_tokens(query))
        ranked = sorted(zip(candidates, scores), key=lambda pair: pair[1], reverse=True)
        return [c for c, score in ranked[:k] if score > 0]

    def external_search(self, query: str, *, k: int = 3) -> list[EvidenceChunk]:
        scores = self._external_bm25.get_scores(lexical_tokens(query))
        ranked = sorted(zip(self.external, scores), key=lambda pair: pair[1], reverse=True)
        return [c for c, score in ranked[:k] if score > 0]

    def authorization_match_exists(self, query: str, principal: Principal, requirements: tuple[str, ...]) -> bool:
        forbidden = [c for c in self.internal if c.metadata["tenant_id"] != principal.tenant_id]
        if not forbidden:
            return False
        candidates = [c for c in forbidden if set(c.metadata["covers"]).intersection(requirements)]
        if not candidates:
            return False
        # This teaching corpus carries explicit requirement labels. In production, an
        # authorization service would return only the existence/reason signal—not content.
        return bool(candidates)


retrievers = LocalRetrievers(internal_corpus, external_corpus)
acme = Principal("user-123", "acme")

dense_identifier = retrievers.dense("AX-774-B", acme, k=3)
lexical_identifier = retrievers.lexical("AX-774-B", acme, k=3)
display(pd.DataFrame({
    "dense": [c.metadata["chunk_id"] for c in dense_identifier],
    "lexical": [c.metadata["chunk_id"] for c in lexical_identifier] + [None] * (3 - len(lexical_identifier)),
}))
assert lexical_identifier[0].metadata["chunk_id"] == "acme-parts-catalog#ax774"


## 5. Stable request-local evidence identity and document-level grading

Retrieval routes return corpus chunks. `merge_evidence` assigns request-local IDs (`E1`, `E2`, …) once and preserves the original chunk, document, source, route, and retrieval query. Query rewrites never replace the original user query in the ledger.

Document-level grading labels candidates as relevant, irrelevant, stale, unauthorized, or duplicate. Set-level grading then asks whether the eligible surviving set covers every requirement and whether authoritative facts conflict.


In [ ]:
def merge_evidence(
    existing: list[EvidenceItem],
    chunks: list[EvidenceChunk],
    *,
    route: str,
    retrieval_query: str,
) -> list[EvidenceItem]:
    merged = list(existing)
    known = {item.chunk.metadata["chunk_id"]: item.evidence_id for item in merged}
    for chunk_ in chunks:
        chunk_id = chunk_.metadata["chunk_id"]
        if chunk_id in known:
            continue
        evidence_id = f"E{len(merged) + 1}"
        merged.append(EvidenceItem(evidence_id, chunk_, route, retrieval_query))
        known[chunk_id] = evidence_id
    return merged


def is_authorized(item: EvidenceItem, principal: Principal, policy: RecoveryPolicy) -> bool:
    metadata_ = item.chunk.metadata
    internal_ok = metadata_["tenant_id"] == principal.tenant_id
    external_ok = metadata_["source_type"] == "external" and policy.external_allowed
    classification_ok = metadata_["classification"] in {"public", principal.clearance}
    return (internal_ok or external_ok) and classification_ok


def grade_documents(
    evidence: list[EvidenceItem],
    requirements: tuple[str, ...],
    principal: Principal,
    policy: RecoveryPolicy,
) -> dict[str, str]:
    labels: dict[str, str] = {}
    seen: set[str] = set()
    for item in evidence:
        chunk_id = item.chunk.metadata["chunk_id"]
        covers = set(item.chunk.metadata["covers"])
        if not is_authorized(item, principal, policy):
            label = "unauthorized"
        elif chunk_id in seen:
            label = "duplicate"
        elif item.chunk.metadata["status"] != "current":
            label = "stale"
        elif covers.intersection(requirements):
            label = "relevant"
        else:
            label = "irrelevant"
        labels[item.evidence_id] = label
        seen.add(chunk_id)
    return labels


## 6. Typed set-level evidence evaluation

The offline evaluator uses transparent teaching labels and real retrieved sets. It does not infer authority or authorization from model prose. Failure diagnosis checks conflict and staleness first, then partial coverage, authorization exclusions, lexical recoverability, semantic rewrite fixtures, and finally corpus absence.

The optional live evaluator receives only already-authorized evidence and must return `EvidenceGrade`. Its output still cannot widen tenant scope, enable an external route, or override budgets.


In [ ]:
REWRITE_FIXTURES = {
    "What continuity maneuver restores the main ledger plane after it stops responding?": "promote warm replica redirect writer endpoint reconciliation checkpoints",
    "How is emergency payment capacity transferred to the reserve data plane?": "checkout failover freeze writes verify replica lag promote warm replica",
    "What elevates an interrupted card purchase to the top urgency class?": "customer-impacting checkout incident escalated Priority A",
}


class FrozenEvidenceEvaluator:
    mode = "frozen_teaching_fixture"

    def grade(
        self,
        *,
        query: str,
        requirements: tuple[str, ...],
        evidence: list[EvidenceItem],
        principal: Principal,
        policy: RecoveryPolicy,
        underspecified: bool = False,
    ) -> EvidenceGrade:
        labels = grade_documents(evidence, requirements, principal, policy)
        relevant_current = [
            item for item in evidence if labels[item.evidence_id] == "relevant"
        ]
        stale_relevant = [
            item for item in evidence
            if labels[item.evidence_id] == "stale"
            and set(item.chunk.metadata["covers"]).intersection(requirements)
        ]
        covered = sorted({
            requirement
            for item in relevant_current
            for requirement in item.chunk.metadata["covers"]
            if requirement in requirements
        })
        missing = [requirement for requirement in requirements if requirement not in covered]

        facts: dict[str, set[str]] = defaultdict(set)
        for item in relevant_current:
            key = item.chunk.metadata.get("fact_key")
            value = item.chunk.metadata.get("fact_value")
            if key and value:
                facts[key].add(value)
        if any(len(values) > 1 for values in facts.values()):
            return EvidenceGrade(state="weak", failure_types=["conflict"], rationale="Eligible current authoritative sources disagree.", covered_requirements=covered, missing_requirements=missing)
        if underspecified:
            return EvidenceGrade(state="insufficient", failure_types=["underspecified"], rationale="A required query parameter is missing.", covered_requirements=covered, missing_requirements=missing)
        if not missing and requirements:
            return EvidenceGrade(state="strong", failure_types=[], rationale="Current authorized evidence covers every explicit requirement.", covered_requirements=covered, missing_requirements=[])
        if stale_relevant and not covered:
            return EvidenceGrade(state="weak", failure_types=["stale"], rationale="Only historical evidence covers the requirement.", covered_requirements=covered, missing_requirements=missing)
        if covered:
            return EvidenceGrade(state="weak", failure_types=["partial_coverage"], rationale="The evidence set covers only part of the compound request.", covered_requirements=covered, missing_requirements=missing)
        if retrievers.authorization_match_exists(query, principal, requirements):
            return EvidenceGrade(state="insufficient", failure_types=["authorization_limited"], rationale="Potentially relevant evidence exists outside the principal's authorized tenant scope.", covered_requirements=[], missing_requirements=missing)
        if query in REWRITE_FIXTURES:
            return EvidenceGrade(state="weak", failure_types=["semantic_gap"], rationale="A bounded retrieval rewrite is available for this vocabulary mismatch.", covered_requirements=[], missing_requirements=missing)
        lexical_candidates = retrievers.lexical(query, principal, k=3)
        if any(set(c.metadata["covers"]).intersection(requirements) for c in lexical_candidates):
            return EvidenceGrade(state="weak", failure_types=["lexical_gap"], rationale="Lexical retrieval can recover an identifier match missed by dense retrieval.", covered_requirements=[], missing_requirements=missing)
        return EvidenceGrade(state="insufficient", failure_types=["corpus_gap"], rationale="No current authorized internal evidence covers the requirement.", covered_requirements=[], missing_requirements=missing)


def build_evaluator():
    live_requested = os.getenv("CRAG_USE_LIVE_GRADER") == "1"
    if live_requested and os.getenv("OPENAI_API_KEY") and os.getenv("CRAG_MODEL"):
        from langchain_openai import ChatOpenAI

        class LiveEvidenceEvaluator:
            mode = "live_structured_output"

            def __init__(self):
                self.model = ChatOpenAI(model=os.environ["CRAG_MODEL"], temperature=0).with_structured_output(EvidenceGrade)

            def grade(self, *, query, requirements, evidence, principal, policy, underspecified=False):
                safe_records = [
                    {"evidence_id": item.evidence_id, "text": item.chunk.text, "metadata": item.chunk.metadata}
                    for item in evidence if is_authorized(item, principal, policy)
                ]
                prompt = (
                    "Grade whether the evidence set covers every requirement. Do not infer facts outside evidence. "
                    f"Query: {query}\nRequirements: {list(requirements)}\nEvidence: {safe_records}"
                )
                return self.model.invoke(prompt)

        return LiveEvidenceEvaluator()
    return FrozenEvidenceEvaluator()


evaluator = build_evaluator()
print("evaluator mode:", evaluator.mode)


## 7. Labelled controller dataset and grader calibration

The same 26 cases drive grader calibration, fixed-RAG evaluation, and corrective-controller evaluation. Each case declares requirements, expected initial evidence state, expected first route, terminal state, and required source chunk IDs. Slices cover strong retrieval, lexical and semantic mismatch, partial coverage, stale evidence, conflict, corpus gaps, authorization limits, underspecification, and budget exhaustion.


In [ ]:
def C(case_id, slice_, query, requirements, grade, route, terminal, required=(), **kwargs):
    return EvalCase(case_id, slice_, query, tuple(requirements), grade, route, terminal, tuple(required), **kwargs)


evaluation_cases = [
    C("S1", "strong_initial", "Which gateway handles Acme checkout?", ["checkout_gateway"], "strong", "none", "answered", ["acme-checkout-runbook#gateway"]),
    C("S2", "strong_initial", "How long are checkout change audit records retained?", ["audit_retention"], "strong", "none", "answered", ["acme-audit-policy#retention"]),
    C("S3", "strong_initial", "When should checkout workers scale for queue depth?", ["scale_trigger"], "strong", "none", "answered", ["acme-capacity-guide#queue"]),
    C("S4", "strong_initial", "Which tenant policy and collection version fields belong in retrieval cache keys?", ["cache_scope"], "strong", "none", "answered", ["acme-cache-policy#scope"], initial_k=4),
    C("L1", "lexical_gap", "AX-774-B", ["ax774_replacement"], "weak", "lexical_fallback", "answered", ["acme-parts-catalog#ax774"]),
    C("L2", "lexical_gap", "RT-991-C", ["rt991_controller"], "weak", "lexical_fallback", "answered", ["acme-parts-catalog#rt991"]),
    C("L3", "lexical_gap", "ZX-204", ["zx204_calibration"], "weak", "lexical_fallback", "answered", ["acme-parts-catalog#zx204"]),
    C("M1", "semantic_gap", "What continuity maneuver restores the main ledger plane after it stops responding?", ["dr_switch"], "weak", "query_rewrite", "answered", ["acme-dr-guide#replica"]),
    C("M2", "semantic_gap", "How is emergency payment capacity transferred to the reserve data plane?", ["failover_steps"], "weak", "query_rewrite", "answered", ["acme-checkout-runbook#failover"]),
    C("M3", "semantic_gap", "What elevates an interrupted card purchase to the top urgency class?", ["priority_rule"], "weak", "query_rewrite", "answered", ["acme-support-guide#priority"]),
    C("P1", "partial_coverage", "What is the checkout failover procedure and who approves it?", ["failover_steps", "failover_approver"], "weak", "targeted_retrieval", "answered", ["acme-checkout-runbook#failover", "acme-checkout-runbook#approval"], initial_k=1),
    C("P2", "partial_coverage", "During a release, what error-rate condition triggers checkout rollback, and who must be paged?", ["rollback_trigger", "oncall_escalation"], "weak", "targeted_retrieval", "answered", ["acme-release-policy#rollback", "acme-oncall#escalation"], initial_k=1),
    C("P3", "partial_coverage", "Give the enterprise SLA and its incident priority rule.", ["enterprise_sla", "priority_rule"], "weak", "targeted_retrieval", "answered", ["acme-sla#enterprise", "acme-support-guide#priority"], initial_k=1),
    C("T1", "stale", "Is the retired 2024 thirty-minute checkout Severity 1 threshold still effective?", ["severity_threshold"], "weak", "fresh_source", "answered", ["acme-incident-policy-v2#severity"], initial_k=1),
    C("T2", "stale", "Replace the historical thirty-minute total-outage rule with the effective threshold.", ["severity_threshold"], "weak", "fresh_source", "answered", ["acme-incident-policy-v2#severity"], initial_k=1),
    C("C1", "conflict", "When is the database maintenance window?", ["maintenance_window"], "weak", "none", "conflicting_evidence", ["acme-change-calendar#window-a", "acme-database-standard#window-b"], initial_k=4),
    C("C2", "conflict", "State the authoritative database maintenance schedule.", ["maintenance_window"], "weak", "none", "conflicting_evidence", ["acme-change-calendar#window-a", "acme-database-standard#window-b"], initial_k=4),
    C("G1", "corpus_gap", "When does Nimbus-Edge version 3 reach end of support?", ["nimbus_eos"], "insufficient", "external_source", "answered", ["approved-nimbus-advisory#eos"]),
    C("G2", "corpus_gap", "When does Regulation R-17 revision 4 take effect?", ["r17_effective_date"], "insufficient", "external_source", "answered", ["approved-regulator-bulletin#r17"]),
    C("G3", "corpus_gap", "When was the west-region public networking incident?", ["west_incident_date"], "insufficient", "none", "insufficient_evidence", [], external_allowed=False),
    C("A1", "authorization_limited", "What is Project Orion's launch window?", ["orion_launch"], "insufficient", "none", "insufficient_authorized_evidence", []),
    C("A2", "authorization_limited", "How does Globex fail over checkout?", ["globex_failover"], "insufficient", "none", "insufficient_authorized_evidence", []),
    C("U1", "underspecified", "What is the Priority A SLA?", ["customer_tier"], "insufficient", "clarification", "clarification_required", []),
    C("U2", "underspecified", "Which residency rule applies?", ["customer_region"], "insufficient", "clarification", "clarification_required", []),
    C("B1", "budget_exhaustion", "Give the undocumented failover rehearsal owner and evidence deadline.", ["rehearsal_owner", "evidence_deadline"], "weak", "targeted_retrieval", "budget_exhausted", [], initial_k=1),
    C("B2", "budget_exhaustion", "Provide the absent vendor waiver approver and expiry rule.", ["waiver_approver", "waiver_expiry"], "weak", "targeted_retrieval", "budget_exhausted", [], initial_k=1),
]

assert 25 <= len(evaluation_cases) <= 40
display(pd.DataFrame([{
    "case": c.case_id, "slice": c.slice, "expected_grade": c.expected_initial_grade,
    "expected_route": c.expected_route, "expected_terminal": c.expected_terminal_state,
} for c in evaluation_cases]))


## 8. Recovery policy and transparent Python controller

Route selection is deterministic from `EvidenceGrade`, `RecoveryPolicy`, and attempt state. Models may grade or rewrite, but they do not invent tools or permissions. Every recovery increments `attempts`, records the route and query, and returns to the same evaluator.

- `lexical_gap` → BM25 fallback
- `semantic_gap` → one bounded rewrite, then dense retrieval
- `partial_coverage` → retrieve only the missing facet
- `stale` → current-only internal retrieval
- `corpus_gap` → approved external corpus when policy allows
- `authorization_limited` / `conflict` → terminal, never broader retrieval


In [ ]:
REQUIREMENT_QUERIES = {
    "failover_steps": "checkout failover freeze writes replica lag promote warm replica",
    "failover_approver": "checkout failover requires approval Database Reliability Incident Commander",
    "rollback_trigger": "rollback begins checkout error rate five percent",
    "oncall_escalation": "page payments on-call failed checkout probes",
    "enterprise_sla": "enterprise customers Priority A response target",
    "priority_rule": "customer-impacting checkout incident Priority A",
    "severity_threshold": "current policy checkout Severity 1 fifteen minutes",
    "rehearsal_owner": "failover rehearsal owner",
    "evidence_deadline": "failover evidence deadline",
    "waiver_approver": "vendor waiver approver",
    "waiver_expiry": "vendor waiver expiry",
}


def first_allowed_route(grade: EvidenceGrade, policy: RecoveryPolicy, rewrites: int) -> str | None:
    mapping = {
        "lexical_gap": "lexical_fallback",
        "semantic_gap": "query_rewrite",
        "partial_coverage": "targeted_retrieval",
        "stale": "fresh_source",
        "corpus_gap": "external_source",
        "underspecified": "clarification",
    }
    for failure in grade.failure_types:
        route = mapping.get(failure)
        if route == "query_rewrite" and rewrites >= policy.max_rewrites:
            continue
        if route == "external_source" and not policy.external_allowed:
            continue
        if route in policy.allowed_routes:
            return route
    return None


def recover(
    *, route: str, query: str, grade: EvidenceGrade, principal: Principal,
    policy: RecoveryPolicy, evidence: list[EvidenceItem], counters: RuntimeCounters,
) -> tuple[list[EvidenceItem], str]:
    retrieval_query = query
    chunks: list[EvidenceChunk] = []
    if route == "lexical_fallback":
        counters.retrieval_calls += 1
        chunks = retrievers.lexical(query, principal, k=3)
    elif route == "query_rewrite":
        counters.rewrite_calls += 1
        counters.retrieval_calls += 1
        retrieval_query = REWRITE_FIXTURES.get(query, query)
        chunks = retrievers.dense(retrieval_query, principal, k=3, status="current")
    elif route == "targeted_retrieval":
        counters.retrieval_calls += 1
        missing = grade.missing_requirements[0]
        retrieval_query = REQUIREMENT_QUERIES.get(missing, missing.replace("_", " "))
        chunks = retrievers.lexical(retrieval_query, principal, k=2, status="current")
    elif route == "fresh_source":
        counters.retrieval_calls += 1
        missing = grade.missing_requirements[0]
        retrieval_query = REQUIREMENT_QUERIES.get(missing, query)
        chunks = retrievers.lexical(retrieval_query, principal, k=3, status="current")
    elif route == "external_source":
        counters.external_source_calls += 1
        counters.retrieval_calls += 1
        chunks = retrievers.external_search(query, k=3)
    elif route == "clarification":
        return evidence, retrieval_query
    else:
        counters.unauthorized_route_attempts += 1
        return evidence, retrieval_query
    return merge_evidence(evidence, chunks, route=route, retrieval_query=retrieval_query), retrieval_query


def render_grounded_answer(
    query: str, evidence: list[EvidenceItem], requirements: tuple[str, ...], counters: RuntimeCounters
) -> dict:
    counters.generation_calls += 1
    if os.getenv("CRAG_USE_LIVE_GENERATOR") == "1" and os.getenv("OPENAI_API_KEY") and os.getenv("CRAG_MODEL"):
        from langchain_openai import ChatOpenAI

        records = [{
            "evidence_id": item.evidence_id,
            "text": item.selected_span or item.chunk.text,
            "source": item.chunk.metadata["source"],
        } for item in evidence]
        prompt = (
            "Answer only from supplied evidence. Cite every factual claim with an evidence_id. "
            "If evidence conflicts, list the conflict instead of resolving it. "
            f"Query: {query}\nRequirements: {list(requirements)}\nEvidence: {records}"
        )
        model = ChatOpenAI(model=os.environ["CRAG_MODEL"], temperature=0).with_structured_output(GroundedAnswer)
        return model.invoke(prompt).model_dump()

    claims, citations = [], []
    for requirement in requirements:
        match = next(item for item in evidence if requirement in item.chunk.metadata["covers"] and item.chunk.metadata["status"] == "current")
        span = match.selected_span or match.chunk.text
        claims.append(f"{span} [{match.evidence_id}]")
        citations.append(match.evidence_id)
    return {"text": " ".join(claims), "citation_ids": citations, "unresolved_conflicts": []}


def validate_answer(result_terminal: str, grade: EvidenceGrade, answer: dict | None, evidence: list[EvidenceItem], principal: Principal, policy: RecoveryPolicy) -> None:
    if result_terminal != "answered":
        assert answer is None
        return
    assert grade.state == "strong"
    known = {item.evidence_id for item in evidence}
    assert answer is not None and set(answer["citation_ids"]).issubset(known)
    assert answer.get("unresolved_conflicts", []) == []
    assert all(is_authorized(item, principal, policy) for item in evidence)


def run_corrective(case: EvalCase) -> ControllerResult:
    started = perf_counter()
    principal = Principal(f"user-{case.case_id.lower()}", case.principal_tenant)
    policy = RecoveryPolicy(external_allowed=case.external_allowed)
    counters = RuntimeCounters(retrieval_calls=1)
    evidence = merge_evidence([], retrievers.dense(case.query, principal, k=case.initial_k), route="dense_initial", retrieval_query=case.query)
    underspecified = case.slice == "underspecified"
    counters.grader_calls += 1
    grade = evaluator.grade(query=case.query, requirements=case.requirements, evidence=evidence, principal=principal, policy=policy, underspecified=underspecified)
    initial_grade = grade.model_copy(deep=True)
    attempts = 0
    rewrites = 0
    route_history: list[str] = []
    recovered_ids: list[str] = []
    terminal: TerminalState | None = None
    answer = None

    while terminal is None:
        if grade.state == "strong":
            terminal = "answered"
            answer = render_grounded_answer(case.query, evidence, case.requirements, counters)
            break
        if "conflict" in grade.failure_types:
            terminal = "conflicting_evidence"
            break
        if "authorization_limited" in grade.failure_types:
            terminal = "insufficient_authorized_evidence"
            break
        if "underspecified" in grade.failure_types:
            route_history.append("clarification")
            terminal = "clarification_required"
            break
        if attempts >= policy.max_attempts:
            terminal = "budget_exhausted"
            break

        route = first_allowed_route(grade, policy, rewrites)
        if route is None:
            terminal = "insufficient_evidence"
            break
        before = {item.evidence_id for item in evidence}
        attempts += 1
        route_history.append(route)
        if route == "query_rewrite":
            rewrites += 1
        evidence, _ = recover(route=route, query=case.query, grade=grade, principal=principal, policy=policy, evidence=evidence, counters=counters)
        recovered_ids.extend(item.evidence_id for item in evidence if item.evidence_id not in before)
        counters.grader_calls += 1
        grade = evaluator.grade(query=case.query, requirements=case.requirements, evidence=evidence, principal=principal, policy=policy, underspecified=False)

    unauthorized_evidence = sum(not is_authorized(item, principal, policy) for item in evidence)
    counters.elapsed_ms = (perf_counter() - started) * 1000
    ledger = {
        "query_id": case.case_id,
        "original_query": case.query,
        "initial_evidence_ids": [item.evidence_id for item in evidence if item.route == "dense_initial"],
        "initial_grade": initial_grade.state,
        "failure_types": initial_grade.failure_types,
        "routes_attempted": route_history,
        "recovered_evidence_ids": recovered_ids,
        "evidence_provenance": [{
            "evidence_id": item.evidence_id,
            "chunk_id": item.chunk.metadata["chunk_id"],
            "document_id": item.chunk.metadata["document_id"],
            "source": item.chunk.metadata["source"],
            "route": item.route,
            "retrieval_query": item.retrieval_query,
        } for item in evidence],
        "final_grade": grade.state,
        "terminal_state": terminal,
        "attempts": attempts,
        "unauthorized_evidence_retrieved": unauthorized_evidence,
    }
    result = ControllerResult(case.case_id, initial_grade, grade, terminal, evidence, attempts, route_history, ledger, counters, answer)
    validate_answer(terminal, grade, answer, evidence, principal, policy)
    assert attempts <= policy.max_attempts
    assert counters.unauthorized_route_attempts == 0
    assert unauthorized_evidence == 0
    return result


## 9. Observe success, recovery, security, conflict, and exhaustion

These traces demonstrate the control invariant rather than only final prose. The lexical case recovers an identifier, the authorization case terminates without widening scope, the conflict case refuses generation, and the impossible compound request stops at the attempt budget.


In [ ]:
examples = [next(c for c in evaluation_cases if c.case_id == case_id) for case_id in ["S1", "L1", "A1", "C1", "B1"]]
example_results = [run_corrective(case) for case in examples]
display(pd.DataFrame([{
    "case": result.case_id,
    "initial_grade": result.initial_grade.state,
    "failures": result.initial_grade.failure_types,
    "routes": result.route_history,
    "attempts": result.attempts,
    "final_grade": result.final_grade.state,
    "terminal": result.terminal_state,
    "retrieval_calls": result.counters.retrieval_calls,
} for result in example_results]))

assert example_results[0].route_history == []
assert example_results[1].route_history == ["lexical_fallback"]
assert example_results[2].terminal_state == "insufficient_authorized_evidence"
assert example_results[3].terminal_state == "conflicting_evidence" and example_results[3].answer is None
assert example_results[4].attempts == RecoveryPolicy().max_attempts
assert example_results[4].terminal_state == "budget_exhausted"


## 10. Evidence refinement without losing provenance

Reranking chooses among candidate chunks. Refinement selects useful spans *inside* an accepted chunk. The operation below uses an authored source span, never a free-form summary, and maps `E1-R1` back to `E1`, its original chunk, and source.


In [ ]:
long_chunk = next(c for c in internal_corpus if c.metadata["chunk_id"] == "acme-refund-policy#long")
raw = merge_evidence([], [long_chunk], route="dense_initial", retrieval_query="Who approves large refunds?")

def refine_item(item: EvidenceItem, requirement: str) -> EvidenceItem:
    selected = item.chunk.metadata["selected_spans"].get(requirement)
    if not selected:
        return item
    return EvidenceItem(
        evidence_id=f"{item.evidence_id}-R1",
        chunk=item.chunk,
        route="evidence_refinement",
        retrieval_query=item.retrieval_query,
        parent_evidence_id=item.evidence_id,
        selected_span=selected,
    )

refined = refine_item(raw[0], "refund_approver")
display(pd.DataFrame([{
    "refined_id": refined.evidence_id,
    "parent_id": refined.parent_evidence_id,
    "chunk_id": refined.chunk.metadata["chunk_id"],
    "source": refined.chunk.metadata["source"],
    "selected_span": refined.selected_span,
}]))
assert refined.parent_evidence_id == "E1"
assert refined.chunk.metadata["chunk_id"] == raw[0].chunk.metadata["chunk_id"]


## 11. Calibrate the evidence evaluator

We evaluate initial grades before celebrating controller outcomes. False acceptance—weak or insufficient evidence labelled strong—is the primary risk because it permits unsupported generation. False correction—strong evidence sent to recovery—adds cost and can introduce worse sources.

The offline results measure this frozen teaching evaluator on this labelled synthetic set. They are not a production model benchmark. With a live evaluator configured, rerun the same table and review disagreement cases before changing routing policy.


In [ ]:
initial_results = []
for case in evaluation_cases:
    principal = Principal(f"cal-{case.case_id}", case.principal_tenant)
    policy = RecoveryPolicy(external_allowed=case.external_allowed)
    evidence = merge_evidence([], retrievers.dense(case.query, principal, k=case.initial_k), route="dense_initial", retrieval_query=case.query)
    predicted = evaluator.grade(query=case.query, requirements=case.requirements, evidence=evidence, principal=principal, policy=policy, underspecified=case.slice == "underspecified")
    initial_results.append({
        "case": case.case_id, "slice": case.slice, "expected": case.expected_initial_grade,
        "predicted": predicted.state, "failure_types": predicted.failure_types,
    })
grader_df = pd.DataFrame(initial_results)
grade_accuracy = (grader_df.expected == grader_df.predicted).mean()
false_accept_rate = ((grader_df.expected != "strong") & (grader_df.predicted == "strong")).mean()
false_correction_rate = ((grader_df.expected == "strong") & (grader_df.predicted != "strong")).mean()
grader_metrics = {"grade_accuracy": grade_accuracy, "false_accept_rate": false_accept_rate, "false_correction_rate": false_correction_rate}
display(pd.Series(grader_metrics, name="measured").to_frame())
display(grader_df[grader_df.expected != grader_df.predicted])


## 12. Fixed-RAG baseline and corrective evaluation

Both systems run on the same cases and first-stage retriever. The fixed baseline retrieves once and answers whenever it receives any candidate—even if the set is stale, conflicting, incomplete, or merely topically similar. CRAG generates only after a strong grade. This makes added retrieval/grading cost visible alongside support and abstention behavior.


In [ ]:
def run_fixed(case: EvalCase) -> dict:
    started = perf_counter()
    principal = Principal(f"fixed-{case.case_id}", case.principal_tenant)
    policy = RecoveryPolicy(external_allowed=case.external_allowed)
    evidence = merge_evidence([], retrievers.dense(case.query, principal, k=case.initial_k), route="dense_initial", retrieval_query=case.query)
    grade = evaluator.grade(query=case.query, requirements=case.requirements, evidence=evidence, principal=principal, policy=policy, underspecified=case.slice == "underspecified")
    answered = bool(evidence)
    retrieved_chunk_ids = {item.chunk.metadata["chunk_id"] for item in evidence}
    required_present = set(case.required_evidence_ids).issubset(retrieved_chunk_ids)
    return {
        "case": case.case_id, "slice": case.slice, "terminal": "answered" if answered else "insufficient_evidence",
        "supported": answered and grade.state == "strong" and required_present,
        "unsupported": answered and grade.state != "strong",
        "correct_abstention": (not answered) and case.expected_terminal_state != "answered",
        "false_abstention": (not answered) and case.expected_terminal_state == "answered",
        "required_evidence_present": required_present,
        "retrieval_calls": 1, "grader_calls": 1, "generation_calls": int(answered),
        "elapsed_ms": (perf_counter() - started) * 1000,
    }


fixed_rows = [run_fixed(case) for case in evaluation_cases]
crag_results = [run_corrective(case) for case in evaluation_cases]
crag_rows = []
for case, result in zip(evaluation_cases, crag_results):
    first_route = result.route_history[0] if result.route_history else "none"
    retrieved_chunk_ids = {item.chunk.metadata["chunk_id"] for item in result.evidence}
    required_present = set(case.required_evidence_ids).issubset(retrieved_chunk_ids)
    crag_rows.append({
        "case": case.case_id, "slice": case.slice, "terminal": result.terminal_state,
        "supported": result.terminal_state == "answered" and result.final_grade.state == "strong" and required_present,
        "unsupported": result.terminal_state == "answered" and result.final_grade.state != "strong",
        "correct_abstention": result.terminal_state != "answered" and case.expected_terminal_state != "answered",
        "false_abstention": result.terminal_state != "answered" and case.expected_terminal_state == "answered",
        "route_correct": first_route == case.expected_route,
        "terminal_correct": result.terminal_state == case.expected_terminal_state,
        "recovery_success": bool(result.route_history) and result.final_grade.state == "strong" and required_present,
        "required_evidence_present": required_present,
        "unnecessary_correction": case.expected_initial_grade == "strong" and bool(result.route_history),
        "attempts": result.attempts,
        "retrieval_calls": result.counters.retrieval_calls,
        "grader_calls": result.counters.grader_calls,
        "generation_calls": result.counters.generation_calls,
        "external_calls": result.counters.external_source_calls,
        "unauthorized_route_attempts": result.counters.unauthorized_route_attempts,
        "elapsed_ms": result.counters.elapsed_ms,
    })
fixed_df, crag_df = pd.DataFrame(fixed_rows), pd.DataFrame(crag_rows)

def p95(values):
    ordered = sorted(values)
    return ordered[max(0, math.ceil(0.95 * len(ordered)) - 1)]

comparison = pd.DataFrame([
    {
        "system": "Fixed RAG",
        "supported_answer_rate": fixed_df.supported.mean(),
        "unsupported_answer_rate": fixed_df.unsupported.mean(),
        "correct_abstention_rate": fixed_df.correct_abstention.mean(),
        "false_abstention_rate": fixed_df.false_abstention.mean(),
        "mean_retrieval_calls": fixed_df.retrieval_calls.mean(),
        "mean_grader_calls": fixed_df.grader_calls.mean(),
        "median_local_ms": median(fixed_df.elapsed_ms),
        "p95_local_ms": p95(fixed_df.elapsed_ms),
    },
    {
        "system": "Corrective RAG",
        "supported_answer_rate": crag_df.supported.mean(),
        "unsupported_answer_rate": crag_df.unsupported.mean(),
        "correct_abstention_rate": crag_df.correct_abstention.mean(),
        "false_abstention_rate": crag_df.false_abstention.mean(),
        "mean_retrieval_calls": crag_df.retrieval_calls.mean(),
        "mean_grader_calls": crag_df.grader_calls.mean(),
        "median_local_ms": median(crag_df.elapsed_ms),
        "p95_local_ms": p95(crag_df.elapsed_ms),
    },
])
display(comparison)


## 13. Controller and route-level metrics

A corrective system can fail by correcting too often, choosing the wrong route, accepting weak evidence, abstaining unnecessarily, or exhausting budget. Route-level analysis answers the operational question: **which recovery route actually provides value?** Local elapsed time is measured, not fabricated, but it is only a small-process diagnostic; call counts are the stable cost comparison.


In [ ]:
controller_metrics = {
    "initial_grade_accuracy": grade_accuracy,
    "false_accept_rate": false_accept_rate,
    "unnecessary_correction_rate": crag_df.unnecessary_correction.mean(),
    "recovery_success_rate": crag_df.loc[crag_df.attempts > 0, "recovery_success"].mean(),
    "final_evidence_support_rate": crag_df.supported.mean(),
    "correct_abstention_rate": crag_df.correct_abstention.mean(),
    "false_abstention_rate": crag_df.false_abstention.mean(),
    "route_accuracy": crag_df.route_correct.mean(),
    "terminal_accuracy": crag_df.terminal_correct.mean(),
    "average_attempts": crag_df.attempts.mean(),
    "unauthorized_route_attempts": int(crag_df.unauthorized_route_attempts.sum()),
}
display(pd.Series(controller_metrics, name="measured").to_frame())

route_rows = []
for route in sorted({route for result in crag_results for route in result.route_history}):
    selected = [(case, result) for case, result in zip(evaluation_cases, crag_results) if route in result.route_history]
    route_rows.append({
        "route": route,
        "cases_attempted": len(selected),
        "recovery_successes": sum(result.final_grade.state == "strong" for _, result in selected),
        "failures": sum(result.final_grade.state != "strong" for _, result in selected),
        "average_attempts": sum(result.attempts for _, result in selected) / len(selected),
        "average_extra_calls": sum(result.counters.retrieval_calls - 1 for _, result in selected) / len(selected),
    })
route_df = pd.DataFrame(route_rows)
display(route_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison.set_index("system")[["supported_answer_rate", "unsupported_answer_rate", "correct_abstention_rate"]].plot.bar(ax=axes[0], color=["#16A3A5", "#EF4444", "#2F6BFF"])
axes[0].set_ylim(0, 1)
axes[0].set_title("Measured outcomes on shared cases")
axes[0].tick_params(axis="x", rotation=0)
route_df.set_index("route")[["recovery_successes", "failures"]].plot.bar(ax=axes[1], color=["#16A3A5", "#F59E42"])
axes[1].set_title("Recovery outcomes by route")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

assert controller_metrics["unauthorized_route_attempts"] == 0
assert crag_df.loc[crag_df.slice == "strong_initial", "attempts"].sum() == 0
assert (crag_df.loc[crag_df.slice == "budget_exhaustion", "terminal"] == "budget_exhausted").all()
assert (crag_df.loc[crag_df.slice == "conflict", "terminal"] == "conflicting_evidence").all()


## 14. Stateful Corrective Controller with LangGraph

`StateGraph`, nodes, edges, and conditional edges package the same explicit state machine. This teaching implementation does not use agent-construction APIs. State includes the principal, policy, requirements, evidence, grade, attempts, route history, counters, terminal reason, and answer; conditional routing reads that state rather than asking a model which tool to call.

The crucial recovery edge is:

```text
recover → grade
```

not `recover → generate`.


In [ ]:
class GraphState(TypedDict):
    case: EvalCase
    principal: Principal
    policy: RecoveryPolicy
    requirements: tuple[str, ...]
    evidence: list[EvidenceItem]
    grade: EvidenceGrade | None
    initial_grade: EvidenceGrade | None
    attempts: int
    rewrites: int
    route_history: list[str]
    planned_route: str | None
    next_action: str
    terminal_state: TerminalState | None
    counters: RuntimeCounters
    answer: dict | None


def graph_retrieve(state: GraphState):
    case, principal = state["case"], state["principal"]
    state["counters"].retrieval_calls += 1
    evidence = merge_evidence([], retrievers.dense(case.query, principal, k=case.initial_k), route="dense_initial", retrieval_query=case.query)
    return {"evidence": evidence}


def graph_grade(state: GraphState):
    state["counters"].grader_calls += 1
    grade = evaluator.grade(
        query=state["case"].query, requirements=state["requirements"], evidence=state["evidence"],
        principal=state["principal"], policy=state["policy"],
        underspecified=state["case"].slice == "underspecified" and state["attempts"] == 0,
    )
    return {"grade": grade, "initial_grade": state["initial_grade"] or grade.model_copy(deep=True)}


def graph_decide(state: GraphState):
    grade, policy = state["grade"], state["policy"]
    assert grade is not None
    if grade.state == "strong":
        return {"next_action": "generate", "planned_route": None}
    terminal_map = {
        "conflict": "conflicting_evidence",
        "authorization_limited": "insufficient_authorized_evidence",
        "underspecified": "clarification_required",
    }
    for failure, terminal in terminal_map.items():
        if failure in grade.failure_types:
            routes = state["route_history"] + (["clarification"] if failure == "underspecified" else [])
            return {"next_action": "abstain", "terminal_state": terminal, "route_history": routes, "planned_route": None}
    if state["attempts"] >= policy.max_attempts:
        return {"next_action": "abstain", "terminal_state": "budget_exhausted", "planned_route": None}
    route = first_allowed_route(grade, policy, state["rewrites"])
    if route is None:
        return {"next_action": "abstain", "terminal_state": "insufficient_evidence", "planned_route": None}
    return {"next_action": "recover", "planned_route": route}


def graph_recover(state: GraphState):
    route = state["planned_route"]
    assert route is not None and state["grade"] is not None
    evidence, _ = recover(
        route=route, query=state["case"].query, grade=state["grade"], principal=state["principal"],
        policy=state["policy"], evidence=state["evidence"], counters=state["counters"],
    )
    return {
        "evidence": evidence,
        "attempts": state["attempts"] + 1,
        "rewrites": state["rewrites"] + int(route == "query_rewrite"),
        "route_history": state["route_history"] + [route],
    }


def graph_generate(state: GraphState):
    assert state["grade"] is not None
    answer = render_grounded_answer(state["case"].query, state["evidence"], state["requirements"], state["counters"])
    validate_answer("answered", state["grade"], answer, state["evidence"], state["principal"], state["policy"])
    return {"answer": answer, "terminal_state": "answered"}


def graph_abstain(state: GraphState):
    return {"answer": None}


def after_decide(state: GraphState) -> Literal["generate", "recover", "abstain"]:
    return state["next_action"]  # type: ignore[return-value]


builder = StateGraph(GraphState)
builder.add_node("retrieve", graph_retrieve)
builder.add_node("grade", graph_grade)
builder.add_node("decide", graph_decide)
builder.add_node("recover", graph_recover)
builder.add_node("generate", graph_generate)
builder.add_node("abstain", graph_abstain)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_edge("grade", "decide")
builder.add_conditional_edges("decide", after_decide, {"generate": "generate", "recover": "recover", "abstain": "abstain"})
builder.add_edge("recover", "grade")
builder.add_edge("generate", END)
builder.add_edge("abstain", END)
corrective_graph = builder.compile()


## 15. Verify graph parity with the transparent controller

We run representative direct, recovered, security-terminal, and budget-terminal cases through both implementations. Framework packaging is acceptable only if terminal reason, route history, and attempt count remain inspectable and equivalent.


In [ ]:
graph_rows = []
for case_id in ["S1", "L1", "A1", "B1"]:
    case = next(c for c in evaluation_cases if c.case_id == case_id)
    policy = RecoveryPolicy(external_allowed=case.external_allowed)
    initial_state: GraphState = {
        "case": case, "principal": Principal(f"graph-{case_id}", case.principal_tenant), "policy": policy,
        "requirements": case.requirements, "evidence": [], "grade": None, "initial_grade": None,
        "attempts": 0, "rewrites": 0, "route_history": [], "planned_route": None,
        "next_action": "", "terminal_state": None, "counters": RuntimeCounters(), "answer": None,
    }
    graph_result = corrective_graph.invoke(initial_state)
    manual_result = next(result for result in crag_results if result.case_id == case_id)
    graph_rows.append({
        "case": case_id,
        "manual_terminal": manual_result.terminal_state,
        "graph_terminal": graph_result["terminal_state"],
        "manual_routes": manual_result.route_history,
        "graph_routes": graph_result["route_history"],
        "manual_attempts": manual_result.attempts,
        "graph_attempts": graph_result["attempts"],
    })
graph_parity = pd.DataFrame(graph_rows)
display(graph_parity)
assert (graph_parity.manual_terminal == graph_parity.graph_terminal).all()
assert (graph_parity.manual_routes == graph_parity.graph_routes).all()
assert (graph_parity.manual_attempts == graph_parity.graph_attempts).all()


## 16. Failure analysis and production upgrades

| Teaching implementation | Production upgrade |
|---|---|
| Deterministic hashing vectors | Versioned embedding service and representative retrieval evaluation |
| Frozen requirement/grade fixtures | Calibrated structured evaluator with held-out human labels and drift monitoring |
| In-memory BM25 and corpus | Governed indexes with authorization filters enforced before evidence exposure |
| Synthetic approved external index | Egress policy, source allowlist, freshness/authority checks, injection defense |
| Process-local attempt counter | Durable state, deadlines, idempotency, cancellation, and concurrency control |
| Mean call counts and local latency | Token/cost accounting, percentiles, tracing, SLOs, and per-route alerts |
| Extractive authored refinement span | Source-locator-preserving passage/table extraction with validation |
| Simple citation renderer | Claim-level citation verification and refusal-safe UI |

**Failure lessons**

- External retrieval is not automatically authoritative; it is re-graded for coverage, authority, freshness, and conflict.
- Authorization and lifecycle eligibility are deterministic boundaries, not model judgements.
- Query rewrites are retrieval hypotheses; traces retain both original and rewritten query.
- Corrective policy can be deterministic and bounded even when retrieval, grading, rewriting, or generation use probabilistic models.
- CRAG is not inherently safe, and Agentic RAG is not inherently unsafe. Permissions, policy, implementation, evaluation, and controls determine safety.


## Exercises

1. Add a hybrid dense+BM25 fusion route and compare it with lexical fallback on identifier cases.
2. Add a precedence policy that resolves one conflict by authority and effective date; retain the unresolved case.
3. Create a held-out human-labelled grading set and calculate confusion matrices by failure slice.
4. Add a latency budget to `RecoveryPolicy` and terminate a deliberately slow route.
5. Implement an optional live query rewriter with a typed schema and prove it cannot change tenant scope.
6. Add claim-level citation validation that rejects a generated claim with no supporting evidence ID.
7. Persist the evidence ledger without storing unnecessary raw restricted text.

## Summary

You replaced a two-path mock with an evidence-recovery controller that retrieves, grades, diagnoses, recovers through bounded policy routes, re-grades, and terminates with explicit reason codes. You measured false acceptance, unnecessary correction, route value, abstention, attempts, calls, and local latency against a fixed-RAG baseline. The LangGraph implementation packages the same state machine without turning it into an open-ended agent.

**Deliberately deferred:** unrestricted live web search, Self-RAG implementation, autonomous tool choice, production IAM, persistent graph checkpoints, and production-scale ANN/load testing belong to later security, Agentic RAG, Adaptive RAG, and operations modules.
